In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
================================================================================
 GC_SAP_external_validation.py -- independent validation and robustness of SAP

 PART A  External validation in two independent bulk gastric-cancer cohorts
         (ACRG GSE62254; Yonsei GSE84437), using ONLY frozen objects from the
         discovery project (nothing is re-fitted or re-selected):
           * SAP signature            S0_signatures_spatial.csv  (134 genes, MOFA+ weights)
           * label-free myeloid states (Fate_TAM_C1QC, Fate_TAM_Inflammatory,
                                        Fate_TAM_SPP1, Mono_classical)  -- same file
           * myeloid abundance panel   (Myeloid_abundance)              -- same file
           * stromal / epithelial abundance genes derived from the discovery atlas
             lineage pseudobulk (S0_atlas_lineage_pseudobulk_cpm.csv.gz), SAP genes removed
         Tests mirror the spatial label-free analysis (Fig. 3/4): partial Spearman
         SAP ~ state, adjusted for stromal and myeloid abundance; Fisher-z fixed-effect
         and DerSimonian-Laird random-effects pooling; 500 expression-matched random
         stromal signatures as empirical null.
           H1  SAP is a state axis, not abundance: activated- and quiescent-arm scores
               are negatively correlated after adjustment for stromal abundance.
           H2  directions frozen from discovery (HYP_STATES in fig4_01):
               TAM_C1QC (-), TAM_Inflammatory (+), Mono_classical (+);
               TAM_SPP1 exploratory (two-sided). TAM_Prolif has no label-free
               signature and cannot be tested in bulk.
 PART B  Robustness of the discovery composition models (Fig. 1E), re-using the
         discovery objects: exact reproduction check, + stromal-fraction covariate,
         >=100 myeloid cells, leave-one-tumour-out, bootstrap CI.

 The analysis plan (hypotheses, thresholds, file hashes) is written to
 analysis_plan.json BEFORE any test is run.

 DATA INTEGRITY: the script never generates, simulates or imputes data. Every number comes
 from (i) the project's own frozen result files and (ii) the public GEO series downloaded
 from NCBI. The only random operations are resampling of these real data (bootstrap of real
 samples; random draws of real stromal genes for the empirical null). If any required file
 is missing, a cohort cannot be downloaded, or Fig. 1E is not reproduced exactly, the script
 stops (or omits that part) instead of substituting anything. provenance.json records SHA-256
 hashes of every input, every downloaded GEO file and every output, plus the GSM IDs used.

 RUN:  pip install GEOparse anndata pandas numpy scipy statsmodels matplotlib
       python GC_SAP_external_validation.py
       (GEO data are downloaded automatically; at the end all results are zipped and
        downloaded automatically: Colab -> browser, Jupyter -> link, terminal -> ~/Downloads)
================================================================================
"""
import os, re, json, hashlib, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE = Path(os.environ.get("GC_BASE", "/home/arshia/payanname"))
F_SIGS = BASE / "GC_Fig4_spatial_outputs/tables/S0_signatures_spatial.csv"
F_LIN = BASE / "GC_Fig4_spatial_outputs/tables/S0_atlas_lineage_pseudobulk_cpm.csv.gz"
F_MCP = BASE / "GC_Fig1_outputs/tables/S2_MCP_scores_v3_amb20_F4.csv"
F_FIG1E = BASE / "GC_Fig1_outputs/tables/Fig1E_composition_vs_axis_m3_amb20_F4.csv"
F_MYELOID = BASE / "GC_Fig1_outputs/h5ad/S1e_myeloid_final_v3_m3_amb20_F4.h5ad"
OUT = BASE / "GC_SAP_validation_outputs"
RAW = OUT / "raw"
for d in (OUT, RAW):
    d.mkdir(parents=True, exist_ok=True)

COHORTS = ["GSE62254", "GSE84437"]          # ACRG; Yonsei (SuperSeries -> subseries resolved automatically)
EXTRA_MATRICES = {}                          # optional, e.g. {"TCGA_STAD": "/path/genes_x_samples_log2tpm.tsv"}

SEED = 0
N_RANDOM = 500
N_BOOT = 1000
MIN_TUMOURS = 50                             # a cohort smaller than this is rejected
MIN_COV = 0.5                                # >= 50% of a signature's genes must be measured
STROMA = ["Fibroblast", "Pericyte_SMC", "Endothelial"]
EPITHELIUM = ["Malignant", "Epithelial_NM"]
ABUND_RATIO, ABUND_MIN_CPM, ABUND_N = 4.0, 20.0, 50   # lineage-specific abundance genes

# frozen from discovery (fig4_01 HYP_STATES_FALLBACK); SPP1 exploratory
STATE_SIGS = {"TAM_C1QC": ("Fate_TAM_C1QC", -1), "TAM_Inflammatory": ("Fate_TAM_Inflammatory", +1),
              "Mono_classical": ("Mono_classical", +1), "TAM_SPP1": ("Fate_TAM_SPP1", 0)}
DISCOVERY_STATES = ["Mono_classical", "TAM_Inflammatory", "TAM_C1QC", "TAM_Prolif"]
COMP_MIN_MYELOID, CLR_PSEUDO = 30, 0.5       # identical to Figure 1

EXCLUDE_GENE_RE = re.compile(r"^(MT-|RP[SL]\d|RPLP|HB[ABDEGMQZ]\d?$|MTRNR|IG[HKL][VJCDG]|TR[ABDG][VJC])")
CELL_CYCLE_RE = re.compile(r"^(MKI67|TOP2A|CDK1|BIRC5|TK1|PCLAF|CCNA2|CCNB1|CCNB2|UBE2C|CENP[AEFW]|KIF\d+[A-Z]?|"
                           r"MCM\d|PCNA|TYMS|RRM2|AURK[AB]|PLK1|BUB1B?|CDC20|CDKN3|NUSAP1|TPX2|ZWINT|"
                           r"FOXM1|E2F[1-8]|HMGB2|STMN1|TUBA1B|H2AZ1|MAD2L1|PBK|MELK|DLGAP5|ASPM|DEPDC1|"
                           r"NCAPG2?|SPC2[45]|SHCBP1)$")
HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
rng = np.random.default_rng(SEED)


def log(msg):
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
    print(line, flush=True)
    with open(OUT / "run_log.txt", "a") as fh:
        fh.write(line + "\n")


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def check_inputs():
    """Stop if any real project file is missing -- nothing is ever substituted."""
    req = [F_SIGS, F_LIN, F_MCP, F_FIG1E, F_MYELOID] + [Path(p) for p in EXTRA_MATRICES.values()]
    miss = [str(p) for p in req if not Path(p).is_file()]
    if miss:
        raise FileNotFoundError("required project files not found (set GC_BASE):\n  " + "\n  ".join(miss))


def clean_outputs():
    """Remove results of earlier runs (raw GEO downloads are kept) so only this run is packaged."""
    for f in OUT.iterdir():
        if f.is_file():
            f.unlink()


def write_provenance(cohorts, extra):
    import importlib.metadata as im
    vers = {}
    for pkg in ["numpy", "pandas", "scipy", "statsmodels", "GEOparse", "anndata", "matplotlib"]:
        try:
            vers[pkg] = im.version(pkg)
        except Exception:
            vers[pkg] = None
    pd.concat([pd.DataFrame({"cohort": n, "sample": X.columns, "subseries": b.values})
               for n, (X, b) in cohorts.items()]).to_csv(OUT / "A_samples_used.csv", index=False)
    prov = dict(created=datetime.datetime.now().isoformat(timespec="seconds"), packages=vers,
                project_inputs={str(p): sha256(p) for p in [F_SIGS, F_LIN, F_MCP, F_FIG1E, F_MYELOID]},
                downloaded_files={f.name: dict(sha256=sha256(f), bytes=f.stat().st_size)
                                  for f in sorted(RAW.iterdir()) if f.is_file()},
                cohorts={n: dict(n_tumours=X.shape[1], n_genes=X.shape[0]) for n, (X, _) in cohorts.items()},
                outputs={f.name: sha256(f) for f in sorted(OUT.iterdir())
                         if f.is_file() and f.suffix != ".zip" and f.name != "provenance.json"}, **extra)
    (OUT / "provenance.json").write_text(json.dumps(prov, indent=2))


# ==============================================================================
# 2. FROZEN SIGNATURES AND ABUNDANCE GENE SETS
# ==============================================================================
def load_signatures():
    t = pd.read_csv(F_SIGS)
    sigs = {k: d.set_index("gene")["weight"] for k, d in t.groupby("signature")}
    sap = sigs["SAP"]
    L = pd.read_csv(F_LIN, index_col=0)

    def specific(target):
        tg = [c for c in target if c in L.columns]
        oth = [c for c in L.columns if c not in tg]
        r = L[tg].max(axis=1) / L[oth].max(axis=1).clip(lower=1e-9)
        ok = (r >= ABUND_RATIO) & (L[tg].max(axis=1) >= ABUND_MIN_CPM)
        ok &= ~r.index.isin(sap.index) & ~r.index.str.match(EXCLUDE_GENE_RE) & ~r.index.str.match(CELL_CYCLE_RE)
        return pd.Series(1.0, index=r[ok].sort_values(ascending=False).index[:ABUND_N])

    sigs["Stroma_abundance"] = specific(STROMA)
    sigs["Epithelial_abundance"] = specific(EPITHELIUM)
    # pool for the random null: stroma-enriched (stroma max >= other max), non-SAP genes
    tg = L[STROMA].max(axis=1)
    oth = L[[c for c in L.columns if c not in STROMA]].max(axis=1)
    pool = L.index[(tg >= oth) & ~L.index.isin(sap.index) & ~L.index.str.match(EXCLUDE_GENE_RE)]
    log(f"signatures: SAP {len(sap)} ({(sap > 0).sum()} activated / {(sap < 0).sum()} quiescent); "
        f"stroma {len(sigs['Stroma_abundance'])}, epithelium {len(sigs['Epithelial_abundance'])}, "
        f"null pool {len(pool)} genes")
    return sigs, pd.Index(pool)


# ==============================================================================
# 3. EXTERNAL COHORTS
# ==============================================================================
def hgnc_alias_map():
    """previous/alias symbol -> current approved symbol (arrays use older symbols, e.g. SEPP1->SELENOP)."""
    f = RAW / "hgnc_complete_set.txt"
    try:
        if not f.exists():
            import urllib.request
            urllib.request.urlretrieve(HGNC_URL, f)
        h = pd.read_csv(f, sep="\t", usecols=["symbol", "prev_symbol", "alias_symbol"], dtype=str)
    except Exception as e:
        log(f"  HGNC table unavailable ({e}); symbols used as annotated")
        return {}
    m = {}
    for col in ["alias_symbol", "prev_symbol"]:          # prev overrides alias
        for s, x in zip(h.symbol, h[col]):
            if isinstance(x, str):
                for a in x.split("|"):
                    m[a.strip()] = s
    for s in h.symbol:
        m[s] = s
    return m


def fetch_gse(acc):
    import GEOparse
    gse = GEOparse.get_GEO(geo=acc, destdir=str(RAW), silent=True)
    if len(gse.gsms) == 0:                               # SuperSeries -> subseries
        subs = [r.split(":")[-1].strip() for r in gse.metadata.get("relation", []) if "SuperSeries of" in r]
        log(f"  {acc} is a SuperSeries of {subs}")
        return [g for s in subs for g in fetch_gse(s)]
    return [(acc, gse)]


def gse_to_matrix(acc, gse, alias):
    expr = gse.pivot_samples("VALUE")
    gpl = list(gse.gpls.values())[0]
    tab = gpl.table
    col = next(c for c in ["Gene Symbol", "Symbol", "ILMN_Gene", "GENE_SYMBOL", "gene_symbol"] if c in tab.columns)
    sym = tab.set_index("ID")[col].astype(str)
    sym = sym[~sym.str.contains("///") & (sym != "nan") & (sym != "")]
    sym = sym.map(lambda s: alias.get(s, s))
    expr = expr.loc[expr.index.intersection(sym.index)].apply(pd.to_numeric, errors="coerce")
    if np.nanpercentile(expr.values, 99) > 100:          # not yet log-scale
        expr = np.log2(expr.clip(lower=0) + 1)
    expr["gene"] = sym.reindex(expr.index).values
    expr["mean"] = expr.drop(columns="gene").mean(axis=1)
    expr = expr.sort_values("mean", ascending=False).drop_duplicates("gene").set_index("gene").drop(columns="mean")
    # keep tumour samples only (drop any sample annotated as normal / non-tumour)
    keep = []
    for s in expr.columns:
        md = gse.gsms[s].metadata
        ch = " ".join(" ".join(md.get(k, [])) for k in ["title", "source_name_ch1", "characteristics_ch1"]).lower()
        if re.search(r"\b(normal|non[- ]?tumou?r|adjacent)\b", ch):
            continue
        keep.append(s)
    log(f"  {acc}: platform {gpl.name}, {len(keep)}/{expr.shape[1]} tumour samples, {expr.shape[0]:,} genes")
    return expr[keep]


def load_cohorts(alias):
    cohorts = {}
    for acc in COHORTS:
        try:
            parts = fetch_gse(acc)
            mats = [(sub, gse_to_matrix(sub, g, alias)) for sub, g in parts]
        except Exception as e:
            log(f"  {acc}: download/parsing failed ({e}) -- cohort skipped")
            continue
        genes = sorted(set.intersection(*[set(m.index) for _, m in mats]))
        X = pd.concat([m.loc[genes] for _, m in mats], axis=1)
        batch = pd.Series(np.concatenate([[sub] * m.shape[1] for sub, m in mats]), index=X.columns)
        if X.shape[1] < MIN_TUMOURS:
            log(f"  {acc}: only {X.shape[1]} tumour samples parsed -- cohort rejected")
            continue
        cohorts[acc] = (X, batch)
    for name, path in EXTRA_MATRICES.items():
        X = pd.read_csv(path, sep="\t", index_col=0)
        X.index = X.index.map(lambda s: alias.get(s, s))
        X = X.groupby(level=0).mean()
        if X.shape[1] < MIN_TUMOURS:
            log(f"  {name}: only {X.shape[1]} samples -- rejected")
            continue
        cohorts[name] = (X, pd.Series(name, index=X.columns))
        log(f"  {name}: {X.shape[1]} samples, {X.shape[0]:,} genes (user matrix)")
    if not cohorts:
        raise RuntimeError("no external cohort could be loaded from GEO -- check the internet connection; "
                           "no result is written without real data")
    return cohorts


# ==============================================================================
# 4. SCORING AND TESTS (same definitions as the spatial analysis)
# ==============================================================================
def zscore_rows(X):
    mu, sd = X.mean(axis=1), X.std(axis=1).replace(0, np.nan)
    return X.sub(mu, axis=0).div(sd, axis=0).dropna(how="all")


def weighted_score(Z, w, min_cov=MIN_COV):
    g = w.index.intersection(Z.index)
    if len(g) < 3 or len(g) / max(len(w), 1) < min_cov:
        return None, len(g) / max(len(w), 1)
    ww = w[g]
    return Z.loc[g].mul(ww, axis=0).sum(axis=0) / ww.abs().sum(), len(g) / len(w)


def _resid(y, C):
    C = np.column_stack([np.ones(len(y)), C])
    return y - C @ np.linalg.lstsq(C, y, rcond=None)[0]


def partial_spearman(x, y, covars):
    """Partial Spearman as in fig4_03: rank x, y and every covariate, residualise the ranks on the
    ranked covariates (OLS), correlate the residuals. Bulk samples are independent, so the
    Dutilleul spatial correction is not needed (n_eff = n)."""
    covars = np.asarray(covars, float).reshape(len(x), -1)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(covars).all(axis=1)
    rx, ry = stats.rankdata(x[ok]), stats.rankdata(y[ok])
    k = 0
    if covars.shape[1]:
        Cr = np.column_stack([stats.rankdata(covars[ok, j]) for j in range(covars.shape[1])])
        Cr = Cr[:, Cr.std(axis=0) > 0]
        k = Cr.shape[1]
        rx, ry = _resid(rx, Cr), _resid(ry, Cr)
    else:
        rx, ry = rx - rx.mean(), ry - ry.mean()
    n = int(ok.sum())
    r = float(np.sum(rx * ry) / np.sqrt(np.sum(rx ** 2) * np.sum(ry ** 2)))
    df = max(n - 2 - k, 1)
    rc = np.clip(r, -0.999999, 0.999999)
    t = rc * np.sqrt(df / (1 - rc ** 2))
    return dict(r=r, p=float(2 * stats.t.sf(abs(t), df)), n=n, k=k)


def batch_dummies(batch):
    b = pd.get_dummies(batch, drop_first=True).astype(float)
    return b.values if b.shape[1] else np.empty((len(batch), 0))


def meta(rows):
    """Fisher-z fixed-effect and DerSimonian-Laird random-effects pooling."""
    z = np.array([np.arctanh(np.clip(r["r"], -0.999, 0.999)) for r in rows])
    v = np.array([1.0 / max(1, r["n"] - 3 - r["k"]) for r in rows])
    w = 1 / v
    zf = np.sum(w * z) / w.sum(); sef = np.sqrt(1 / w.sum())
    Q = np.sum(w * (z - zf) ** 2); dfq = len(z) - 1
    tau2 = max(0.0, (Q - dfq) / (w.sum() - (w ** 2).sum() / w.sum())) if dfq > 0 else 0.0
    wr = 1 / (v + tau2); zr = np.sum(wr * z) / wr.sum(); ser = np.sqrt(1 / wr.sum())
    return dict(r_FE=np.tanh(zf), lo_FE=np.tanh(zf - 1.96 * sef), hi_FE=np.tanh(zf + 1.96 * sef),
                p_FE=2 * stats.norm.sf(abs(zf / sef)), r_RE=np.tanh(zr), p_RE=2 * stats.norm.sf(abs(zr / ser)),
                I2=max(0.0, (Q - dfq) / Q) if Q > 0 and dfq > 0 else 0.0, k=len(z),
                n_pos=int((z > 0).sum()), n_neg=int((z < 0).sum()))


def one_sided(p_two, r, expected):
    if expected == 0:
        return p_two
    return p_two / 2 if np.sign(r) == expected else 1 - p_two / 2


def cohort_tests(name, X, batch, sigs, pool):
    Z = zscore_rows(X)
    sc, cov = {}, {}
    for k in ["SAP", "Stroma_abundance", "Epithelial_abundance", "Myeloid_abundance"] + [v[0] for v in STATE_SIGS.values()]:
        sc[k], cov[k] = weighted_score(Z, sigs[k])
    sap = sigs["SAP"]
    sc["SAP_up"], cov["SAP_up"] = weighted_score(Z, sap[sap > 0])
    sc["SAP_down"], cov["SAP_down"] = weighted_score(Z, -sap[sap < 0])
    log(f"  {name}: coverage " + ", ".join(f"{k} {v:.0%}" for k, v in cov.items()))
    miss = [k for k in ["SAP", "Stroma_abundance", "Myeloid_abundance"] if sc[k] is None]
    if miss:
        raise RuntimeError(f"{name}: insufficient gene coverage for {miss}")
    B = batch_dummies(batch)
    S, M, E = (sc[k].values for k in ["Stroma_abundance", "Myeloid_abundance", "Epithelial_abundance"])
    rows = []

    def add(analysis, state, x, y, C, expected):
        C = np.column_stack([C, B]) if B.size else C
        r = partial_spearman(x, y, C)
        rows.append(dict(cohort=name, analysis=analysis, state=state, expected=expected, **r))
        return rows[-1], C

    # H1: arms anti-correlated after stromal adjustment (state axis, not abundance)
    add("H1 activated vs quiescent arm | stroma", "SAP arms", sc["SAP_up"].values, sc["SAP_down"].values,
        S[:, None], -1)
    add("H1 (unadjusted)", "SAP arms", sc["SAP_up"].values, sc["SAP_down"].values, np.empty((len(S), 0)), 0)
    # H2 and sensitivities
    for state, (sig, exp) in STATE_SIGS.items():
        if sc[sig] is None:
            log(f"  {name}: {state} not tested (coverage {cov[sig]:.0%})")
            continue
        y, x = sc[sig].values, sc["SAP"].values
        prim, C = add("H2 SAP vs state | stroma, myeloid", state, x, y, np.column_stack([S, M]), exp)
        # bootstrap 95% CI of the primary estimate (samples resampled with replacement)
        bs = [partial_spearman(x[i], y[i], C[i])["r"]
              for i in (rng.integers(0, len(y), len(y)) for _ in range(N_BOOT))]
        prim.update(boot_lo=np.nanpercentile(bs, 2.5), boot_hi=np.nanpercentile(bs, 97.5))
        if sc["Epithelial_abundance"] is not None:
            add("Sens: + epithelial abundance", state, x, y, np.column_stack([S, M, E]), exp)
        add("Sens: raw SAP (no stromal adjustment)", state, x, y, M[:, None], exp)
        add("Sens: activated arm only", state, sc["SAP_up"].values, y, np.column_stack([S, M]), exp)
        add("Sens: quiescent arm only (reversed)", state, -sc["SAP_down"].values, y, np.column_stack([S, M]), exp)
    res = pd.DataFrame(rows)
    # random stromal null for the primary H2 test (expression-matched, SAP weights)
    nul = random_null(name, Z, X, sigs, pool, S, M, B)
    return res, nul, pd.DataFrame({k: v for k, v in sc.items() if v is not None})


def random_null(name, Z, X, sigs, pool, S, M, B):
    sap = sigs["SAP"]
    sg = sap.index.intersection(Z.index)
    pl = pool.intersection(Z.index)
    mean = X.mean(axis=1)
    edges = np.unique(np.quantile(mean.loc[sg.union(pl)], np.linspace(0, 1, 11)))
    bin_of = lambda g: np.clip(np.searchsorted(edges, mean.loc[g].values, side="right") - 1, 0, len(edges) - 2)
    bp, bs = bin_of(pl), bin_of(sg)
    by_bin = {b: pl[bp == b] for b in np.unique(bp)}
    C = np.column_stack([S, M, B]) if B.size else np.column_stack([S, M])
    ys = {st: weighted_score(Z, sigs[sig])[0] for st, (sig, _) in STATE_SIGS.items()}
    out = []
    for d in range(N_RANDOM):
        r_ = np.random.default_rng(SEED + d)
        # one random stromal gene per SAP gene, from the same expression decile, carrying that SAP weight
        genes = [r_.choice(by_bin[b]) if b in by_bin and len(by_bin[b]) else r_.choice(pl) for b in bs]
        w = pd.Series(sap[sg].values, index=pd.Index(genes)).groupby(level=0).mean()
        x, _ = weighted_score(Z, w, min_cov=0.3)
        if x is None:
            continue
        for state, y in ys.items():
            if y is not None:
                out.append(dict(cohort=name, draw=d, state=state, r=partial_spearman(x.values, y.values, C)["r"]))
    return pd.DataFrame(out)


# ==============================================================================
# 5. PART B -- ROBUSTNESS OF THE DISCOVERY COMPOSITION MODELS (Fig. 1E)
# ==============================================================================
def clr(counts, pseudo=CLR_PSEUDO):
    L = np.log(counts + pseudo)
    return L.sub(L.mean(axis=1), axis=0)


def comp_fit(counts, meta_, rhs):
    Y = clr(counts)          # not named C: patsy would shadow C() in the formula
    m = meta_.loc[counts.index].copy()
    m["MCP_z"] = (m.MCP - m.MCP.mean()) / m.MCP.std(ddof=1)
    out = {}
    for s in counts.columns:
        if (counts[s] > 0).sum() < 5:
            continue
        f = smf.ols(f"y ~ {rhs}", data=m.assign(y=Y[s].values)).fit()
        out[s] = (f.params["MCP_z"], f.bse["MCP_z"], f.pvalues["MCP_z"], len(m))
    return pd.DataFrame(out, index=["beta", "se", "p", "n"]).T


def discovery_robustness():
    import anndata as ad
    my = ad.read_h5ad(F_MYELOID, backed="r")
    obs = my.obs[["sample_id", "myeloid_state"]].copy()
    my.file.close()
    mcp = pd.read_csv(F_MCP, index_col=0)
    counts = obs.groupby(["sample_id", "myeloid_state"], observed=False).size().unstack(fill_value=0)
    counts = counts.loc[counts.sum(axis=1) >= COMP_MIN_MYELOID]
    counts = counts.loc[counts.index.intersection(mcp.index)]
    rows = []
    base = comp_fit(counts, mcp, "C(dataset) + MCP_z")
    ref = pd.read_csv(F_FIG1E).set_index("state")
    diff = (base.beta - ref.beta_clr_per_SD.reindex(base.index)).abs().max()
    n_ref = int(ref.n_samples.iloc[0])
    log(f"PART B reproduction of Fig. 1E: n = {len(counts)} (published {n_ref}), "
        f"max |beta difference| = {diff:.2e}")
    if not (diff <= 1e-6 and len(counts) == n_ref):
        log("  ERROR: Fig. 1E is not reproduced exactly from these inputs -- Part B is NOT written")
        return None
    for label, cnt, rhs in [("Fig. 1E (reproduced)", counts, "C(dataset) + MCP_z"),
                            ("+ stromal fraction", counts, "C(dataset) + MCP_z + stromal_fraction"),
                            (">=100 myeloid cells", counts.loc[counts.sum(axis=1) >= 100], "C(dataset) + MCP_z")]:
        f = comp_fit(cnt, mcp, rhs)
        f["fdr_within"] = multipletests(f.p, method="fdr_bh")[1]
        for s in DISCOVERY_STATES:
            if s in f.index:
                rows.append(dict(analysis=label, state=s, **f.loc[s].to_dict()))
    # leave-one-tumour-out and bootstrap for the four states
    loo = {s: [] for s in DISCOVERY_STATES}
    for t in counts.index:
        f = comp_fit(counts.drop(index=t), mcp, "C(dataset) + MCP_z")
        for s in DISCOVERY_STATES:
            if s in f.index:
                loo[s].append(f.loc[s, "beta"])
    bt = {s: [] for s in DISCOVERY_STATES}
    for b in range(N_BOOT):
        idx = rng.choice(counts.index, len(counts), replace=True)
        cb = counts.loc[idx].copy()
        cb.index = [f"{i}_{j}" for j, i in enumerate(idx)]
        mb = mcp.loc[idx].copy(); mb.index = cb.index
        if mb.dataset.nunique() < 2:
            continue
        f = comp_fit(cb, mb, "C(dataset) + MCP_z")
        for s in DISCOVERY_STATES:
            if s in f.index:
                bt[s].append(f.loc[s, "beta"])
    for s in DISCOVERY_STATES:
        if loo[s]:
            b0 = base.loc[s, "beta"]
            rows.append(dict(analysis="leave-one-tumour-out", state=s, beta=b0, beta_min=np.min(loo[s]),
                             beta_max=np.max(loo[s]), same_sign_frac=np.mean(np.sign(loo[s]) == np.sign(b0)),
                             n=len(counts)))
            rows.append(dict(analysis="bootstrap 95% CI", state=s, beta=b0, ci_lo=np.percentile(bt[s], 2.5),
                             ci_hi=np.percentile(bt[s], 97.5), n=len(counts)))
    res = pd.DataFrame(rows)
    res.to_csv(OUT / "B_discovery_composition_robustness.csv", index=False)
    return res


# ==============================================================================
# 6. MAIN
# ==============================================================================
def write_plan():
    plan = dict(created=datetime.datetime.now().isoformat(timespec="seconds"),
                cohorts=COHORTS + list(EXTRA_MATRICES),
                hypotheses={"H1": "activated vs quiescent SAP arm, partial Spearman | stromal abundance < 0",
                            "H2": {k: v[1] for k, v in STATE_SIGS.items()}},
                primary="FE-pooled partial Spearman, one-sided in the expected direction, BH over the 3 "
                        "directional states; replication = q < 0.05 AND same sign in every cohort",
                secondary="RE P, per-cohort estimates, 500-draw random stromal null, sensitivities",
                not_testable="TAM_Prolif (no label-free signature; bulk proliferation dominated by tumour cells)",
                inputs={str(p): sha256(p) for p in [F_SIGS, F_LIN, F_MCP, F_FIG1E]},
                params=dict(SEED=SEED, N_RANDOM=N_RANDOM, N_BOOT=N_BOOT, MIN_COV=MIN_COV,
                            ABUND_RATIO=ABUND_RATIO, ABUND_MIN_CPM=ABUND_MIN_CPM, ABUND_N=ABUND_N))
    (OUT / "analysis_plan.json").write_text(json.dumps(plan, indent=2))
    log("analysis plan frozen -> analysis_plan.json")


def summarise(res, nul):
    out = []
    for (an, st), d in res.groupby(["analysis", "state"]):
        m = meta(d.to_dict("records"))
        exp = int(d.expected.iloc[0])
        m.update(analysis=an, state=st, expected=exp, p_FE_one_sided=one_sided(m["p_FE"], m["r_FE"], exp),
                 per_cohort="; ".join(f"{r.cohort} {r.r:+.3f} (P={r.p:.2g}, n={r.n})" for r in d.itertuples()))
        out.append(m)
    s = pd.DataFrame(out)
    prim = s.analysis.str.startswith("H2") & (s.expected != 0)
    s.loc[prim, "q_primary"] = multipletests(s.loc[prim, "p_FE_one_sided"], method="fdr_bh")[1]
    s.loc[prim, "replicated"] = (s.loc[prim, "q_primary"] < 0.05) & \
        (np.where(s.loc[prim, "expected"] > 0, s.loc[prim, "n_neg"], s.loc[prim, "n_pos"]) == 0)
    if len(nul):
        obs = res[res.analysis.str.startswith("H2")].groupby("state").apply(
            lambda d: meta(d.to_dict("records"))["r_FE"])
        nm = nul.groupby(["draw", "state"]).r.apply(lambda r: np.tanh(np.mean(np.arctanh(r)))).unstack()
        for st in nm.columns:
            e = STATE_SIGS[st][1] or np.sign(obs[st])
            pe = (1 + (e * nm[st] >= e * obs[st]).sum()) / (1 + nm[st].notna().sum())
            s.loc[s.analysis.str.startswith("H2") & (s.state == st), "P_emp_random_stroma"] = pe
    s.to_csv(OUT / "A_external_validation_summary.csv", index=False)
    return s


def forest(res, s):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = res[res.analysis.str.startswith("H2")]
    states = list(STATE_SIGS)
    fig, axes = plt.subplots(1, len(states), figsize=(2.3 * len(states), 2.4), sharey=True)
    for ax, st in zip(axes, states):
        dd = d[d.state == st]
        for i, r in enumerate(dd.itertuples()):
            se = 1 / np.sqrt(max(1, r.n - 3 - r.k))
            lo, hi = np.tanh(np.arctanh(r.r) - 1.96 * se), np.tanh(np.arctanh(r.r) + 1.96 * se)
            ax.plot([lo, hi], [i, i], color="0.4"); ax.plot(r.r, i, "s", color="k")
        m = s[(s.state == st) & s.analysis.str.startswith("H2")]
        if len(m):
            m = m.iloc[0]
            ax.plot([m.lo_FE, m.hi_FE], [len(dd)] * 2, color="C3", lw=3)
        ax.axvline(0, ls=":", color="0.5")
        ax.set_yticks(range(len(dd) + 1)); ax.set_yticklabels(list(dd.cohort) + ["FE meta"], fontsize=7)
        exp = STATE_SIGS[st][1]
        ax.set_title(f"{st}\n(expected {'+' if exp > 0 else '-' if exp < 0 else 'n/a'})", fontsize=8)
        ax.set_xlabel("partial Spearman ρ", fontsize=7)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(OUT / f"A_external_validation_forest.{ext}", dpi=300)


def package_and_download():
    """Zip every result (tables, figures, plan, log; the large raw GEO downloads are left out) and
    deliver it automatically: Colab -> browser download; Jupyter -> download link;
    terminal -> copy to ~/Downloads (or DOWNLOAD_DIR)."""
    import zipfile, shutil
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    zp = OUT / f"GC_SAP_validation_results_{stamp}.zip"
    files = [f for f in OUT.rglob("*") if f.is_file() and RAW not in f.parents and f.suffix != ".zip"]
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, f.relative_to(OUT))
    log(f"results packed: {zp} ({len(files)} files, {zp.stat().st_size / 1e6:.1f} MB)")
    try:                                              # Google Colab
        from google.colab import files as colab_files
        colab_files.download(str(zp))
        log("download started in the browser (Colab)")
        return zp
    except Exception:
        pass
    try:                                              # Jupyter notebook / lab
        from IPython import get_ipython
        if get_ipython() is not None and "IPKernelApp" in get_ipython().config:
            from IPython.display import FileLink, display
            display(FileLink(os.path.relpath(zp), result_html_prefix="Download results: "))
            log("download link shown in the notebook")
    except Exception:
        pass
    dl = Path(os.environ.get("DOWNLOAD_DIR", Path.home() / "Downloads"))
    try:                                              # terminal / script run
        dl.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zp, dl / zp.name)
        log(f"results copied to {dl / zp.name}")
    except Exception as e:
        log(f"could not copy to {dl} ({e}); the zip is at {zp}")
    return zp


def main():
    check_inputs()
    clean_outputs()
    write_plan()
    sigs, pool = load_signatures()
    alias = hgnc_alias_map()
    cohorts = load_cohorts(alias)
    allres, allnul, scores = [], [], []
    for name, (X, batch) in cohorts.items():
        r, n, sc = cohort_tests(name, X, batch, sigs, pool)
        allres.append(r); allnul.append(n); scores.append(sc.assign(cohort=name))
    res, nul = pd.concat(allres), pd.concat(allnul)
    res.to_csv(OUT / "A_external_validation_per_cohort.csv", index=False)
    pd.concat(scores).to_csv(OUT / "A_external_scores_per_sample.csv")
    s = summarise(res, nul)
    forest(res, s)
    b = discovery_robustness()
    with open(OUT / "methods_numbers_validation.txt", "w") as fh:
        for name, (X, _) in cohorts.items():
            fh.write(f"{name}: {X.shape[1]} tumours, {X.shape[0]} genes\n")
        cols = ["analysis", "state", "expected", "r_FE", "lo_FE", "hi_FE", "p_FE", "p_FE_one_sided",
                "q_primary", "replicated", "p_RE", "I2", "n_pos", "n_neg", "P_emp_random_stroma", "per_cohort"]
        fh.write(s[[c for c in cols if c in s]].to_string(index=False) + "\n")
        if b is not None:
            fh.write("\nPART B discovery robustness\n" + b.to_string(index=False) + "\n")
    write_provenance(cohorts, dict(part_B_written=b is not None))
    log(f"done -> {OUT}")
    package_and_download()


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
================================================================================
 GC_SAP_validation_figure.py -- multi-panel figure for the independent validation

 Reads ONLY the real outputs written by GC_SAP_external_validation.py
 (BASE/GC_SAP_validation_outputs) plus the frozen signature file. Nothing is
 simulated; every plotted number is read from those files, and the partial-rank
 residual scatters are re-derived from the per-sample scores and checked against
 the reported coefficients (the script stops if they disagree).

   A  study design (numbers read from the result files)
   B  H1 - SAP arms co-vary with stromal abundance but oppose each other once
          abundance is removed (state axis, not abundance)
   C  H2 - forest plot, per-cohort and fixed-effect pooled partial Spearman
   D  partial-rank residual scatters (SAP vs state | stroma, myeloid)
   E  sensitivity analyses (heat map of partial Spearman)
   F  robustness of the discovery composition models (Fig. 1E)

 RUN:  python GC_SAP_validation_figure.py
 OUT:  BASE/GC_SAP_validation_outputs/figures/Fig_SAP_external_validation.{pdf,svg,png,tiff}
================================================================================
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Polygon
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

BASE = Path(os.environ.get("GC_BASE", "/home/arshia/payanname"))
RES = BASE / "GC_SAP_validation_outputs"
F_SIGS = BASE / "GC_Fig4_spatial_outputs/tables/S0_signatures_spatial.csv"
FIG = RES / "figures"
FIG.mkdir(parents=True, exist_ok=True)

need = {k: RES / v for k, v in dict(summary="A_external_validation_summary.csv",
                                    cohort="A_external_validation_per_cohort.csv",
                                    scores="A_external_scores_per_sample.csv",
                                    disc="B_discovery_composition_robustness.csv").items()}
miss = [str(p) for p in list(need.values()) + [F_SIGS] if not p.is_file()]
if miss:
    raise FileNotFoundError("run GC_SAP_external_validation.py first; missing:\n  " + "\n  ".join(miss))

S = pd.read_csv(need["summary"])
P = pd.read_csv(need["cohort"])
SC = pd.read_csv(need["scores"], index_col=0)
D = pd.read_csv(need["disc"])
SIG = pd.read_csv(F_SIGS)

# ------------------------------------------------------------------ style
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5,
    "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", "savefig.dpi": 600})

COH = {"GSE62254": dict(label="ACRG", color="#2F6DB5"), "GSE84437": dict(label="Yonsei", color="#E07A2E")}
COH_ORDER = [c for c in COH if c in P.cohort.unique()]
STATE = {"TAM_C1QC": dict(label="TAM C1QC", exp=-1, sig="Fate_TAM_C1QC"),
         "TAM_Inflammatory": dict(label="TAM Inflammatory", exp=+1, sig="Fate_TAM_Inflammatory"),
         "Mono_classical": dict(label="Classical monocyte", exp=+1, sig="Mono_classical"),
         "TAM_SPP1": dict(label="TAM SPP1", exp=0, sig="Fate_TAM_SPP1")}
POS, NEG, GREY, INK = "#C0392B", "#2471A3", "#8C8C8C", "#1F1F1F"
DIV = LinearSegmentedColormap.from_list("div", ["#1F4E8C", "#6FA3D2", "#F4F4F4", "#E58B6D", "#A3222A"])


def fmt_p(p):
    if not np.isfinite(p):
        return "n/a"
    if p < 1e-300:
        return "<1e-300"
    if p < 1e-3:
        m, e = f"{p:.1e}".split("e")
        return f"{m}×10$^{{{int(e)}}}$"
    return f"{p:.3f}" if p < 0.1 else f"{p:.2f}"


def fisher_ci(r, n, k):
    se = 1 / np.sqrt(max(1, n - 3 - k))
    z = np.arctanh(np.clip(r, -0.999, 0.999))
    return np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)


def panel_letter(ax, s, dx=-0.0, dy=0.0):
    ax.text(dx, 1.0 + dy, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom", ha="right")


def rank_resid(v, C):
    r = stats.rankdata(v)
    Cr = np.column_stack([np.ones(len(v))] + [stats.rankdata(C[:, j]) for j in range(C.shape[1])])
    return r - Cr @ np.linalg.lstsq(Cr, r, rcond=None)[0]


def prim(state, cohort=None):
    d = P[(P.analysis == "H2 SAP vs state | stroma, myeloid") & (P.state == state)]
    return d if cohort is None else d[d.cohort == cohort].iloc[0]


def meta_row(analysis, state):
    d = S[(S.analysis == analysis) & (S.state == state)]
    return d.iloc[0] if len(d) else None


# ------------------------------------------------------------------ layout (180 mm wide)
fig = plt.figure(figsize=(7.2, 9.6))
outer = gridspec.GridSpec(4, 1, figure=fig, height_ratios=[0.5, 1.2, 1.25, 1.35], hspace=0.34,
                          left=0.07, right=0.985, top=0.98, bottom=0.085)

# ================================================================== A  design
axA = fig.add_subplot(outer[0])
axA.set_xlim(0, 100); axA.set_ylim(0.5, 19.5); axA.axis("off")
panel_letter(axA, "A", dx=0.0, dy=-0.02)
sap = SIG[SIG.signature == "SAP"]
n_up, n_dn = int((sap.weight > 0).sum()), int((sap.weight < 0).sum())
n_disc = int(D[D.analysis == "Fig. 1E (reproduced)"].n.iloc[0])
ncoh = {c: int(P[P.cohort == c].n.max()) for c in COH_ORDER}
boxes = [
    (1, "Discovery",
     f"scRNA-seq: GSE150290, GSE183904\nSAP: {len(sap)} genes ({n_up} up / {n_dn} down)\n"
     f"myeloid state signatures\ncomposition: n = {n_disc} tumours"),
    (35, "Validation cohorts",
     "\n".join(f"{COH[c]['label']} ({c}): n = {ncoh[c]}" for c in COH_ORDER) +
     "\nsignatures fixed (no re-fitting)"),
    (69, "Tests",
     "H1: SAP arms | stroma\nH2: SAP vs state | stroma, myeloid\nfixed/random-effects meta-analysis\n"
     "500 random stromal signatures"),
]
for x, title, body in boxes:
    axA.add_patch(plt.Rectangle((x, 1.2), 30, 17.3, fc="white", ec="black", lw=0.6))
    axA.text(x + 15, 16.3, title, ha="center", va="center", fontsize=7, fontweight="bold", color="black")
    axA.text(x + 15, 7.9, body, ha="center", va="center", fontsize=6.2, color="black", linespacing=1.35)
for x in (31.3, 65.3):
    axA.annotate("", xy=(x + 3.2, 9.8), xytext=(x, 9.8),
                 arrowprops=dict(arrowstyle="->", lw=0.7, color="black"))

# ================================================================== B / C
rowBC = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[1], width_ratios=[0.8, 1.5, 0.62], wspace=0.08)

# ---- B: H1 dumbbell
axB = fig.add_subplot(rowBC[0])
axB.set_position(axB.get_position().translated(-0.0, 0))
panel_letter(axB, "B", dx=-0.16)
ys = np.arange(len(COH_ORDER))[::-1] + 1
for y, c in zip(ys, COH_ORDER):
    u = P[(P.cohort == c) & (P.analysis == "H1 (unadjusted)")].iloc[0]
    a = P[(P.cohort == c) & (P.analysis == "H1 activated vs quiescent arm | stroma")].iloc[0]
    col = COH[c]["color"]
    axB.annotate("", xy=(a.r, y), xytext=(u.r, y),
                 arrowprops=dict(arrowstyle="-|>", color=col, lw=1.3, mutation_scale=8, shrinkA=3, shrinkB=3))
    axB.plot(u.r, y, "o", ms=5.5, mfc="white", mec=col, mew=1.2, zorder=3)
    axB.plot(a.r, y, "o", ms=5.5, color=col, zorder=3)
    lo, hi = fisher_ci(a.r, a.n, a.k)
    axB.plot([lo, hi], [y - 0.18] * 2, color=col, lw=0.8)
m_u, m_a = meta_row("H1 (unadjusted)", "SAP arms"), meta_row("H1 activated vs quiescent arm | stroma", "SAP arms")
axB.plot(m_u.r_FE, 0, "D", ms=5, mfc="white", mec=INK, mew=1.1)
axB.plot(m_a.r_FE, 0, "D", ms=5, color=INK)
axB.annotate("", xy=(m_a.r_FE, 0), xytext=(m_u.r_FE, 0),
             arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3, mutation_scale=8, shrinkA=3, shrinkB=3))
axB.axvline(0, color=GREY, lw=0.6, ls=(0, (2, 2)))
axB.set_yticks(list(ys) + [0]); axB.set_yticklabels([COH[c]["label"] for c in COH_ORDER] + ["FE meta"])
axB.set_ylim(-0.7, len(COH_ORDER) + 0.7)
axB.set_xlabel("Spearman ρ, activated vs quiescent arm")
axB.set_title("H1  SAP is a state axis", loc="left", fontweight="bold")
axB.text(0.02, 0.985, f"adjusted FE ρ = {m_a.r_FE:.2f}\nP = {fmt_p(m_a.p_FE)}, I² = {m_a.I2:.0%}",
         transform=axB.transAxes, fontsize=5.8, va="top", color=INK,
         bbox=dict(fc="white", ec="none", pad=0.6))
axB.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec=INK, ms=5, label="unadjusted"),
                    Line2D([], [], marker="o", ls="", color=INK, ms=5, label="| stromal abundance")],
           loc="lower right", fontsize=5.8, handletextpad=0.2, borderaxespad=0.1)

# ---- C: H2 forest
axC = fig.add_subplot(rowBC[1])
axT = fig.add_subplot(rowBC[2], sharey=axC)
axT.axis("off")
pc = axC.get_position(); axC.set_position([pc.x0 + 0.075, pc.y0, pc.width - 0.075, pc.height])
panel_letter(axC, "C", dx=-0.3)
y, yt, ylab = 0, [], []
order = ["TAM_C1QC", "TAM_Inflammatory", "Mono_classical", "TAM_SPP1"]
gap = 0.9
for st in order:
    info, mm = STATE[st], meta_row("H2 SAP vs state | stroma, myeloid", st)
    base_col = POS if info["exp"] > 0 else NEG if info["exp"] < 0 else GREY
    top = y
    for c in COH_ORDER:
        r = prim(st, c)
        lo, hi = (r.boot_lo, r.boot_hi) if np.isfinite(r.get("boot_lo", np.nan)) else fisher_ci(r.r, r.n, r.k)
        axC.plot([lo, hi], [y, y], color=COH[c]["color"], lw=1.2, solid_capstyle="round")
        axC.plot(r.r, y, "s", ms=3 + 3 * np.sqrt(r.n / 500), color=COH[c]["color"], mec="white", mew=0.4, zorder=3)
        yt.append(y); ylab.append(f"{COH[c]['label']}  (n = {r.n})")
        y -= 1
    d = Polygon([[mm.lo_FE, y], [mm.r_FE, y + 0.3], [mm.hi_FE, y], [mm.r_FE, y - 0.3]], closed=True,
                fc=base_col, ec=base_col, lw=0.6, zorder=3)
    axC.add_patch(d)
    yt.append(y); ylab.append("FE meta")
    axC.axhspan(y - 0.55, top + 1.15, color=base_col, alpha=0.06, lw=0, zorder=0)
    exp_txt = {1: "expected +", -1: "expected −", 0: "exploratory"}[info["exp"]]
    axC.text(-0.79, top + 0.75, f"{info['label']}  ·  {exp_txt}", fontsize=6.6, fontweight="bold",
             color=base_col, va="center")
    q = mm.get("q_primary", np.nan)
    stat = (f"ρ = {mm.r_FE:+.2f} [{mm.lo_FE:+.2f}, {mm.hi_FE:+.2f}]\n"
            + (f"q = {fmt_p(q)}" if np.isfinite(q) else f"P = {fmt_p(mm.p_FE)}")
            + f";  P$_{{null}}$ {'<' if mm.P_emp_random_stroma <= 1 / 500 + 1e-9 else '='} "
              f"{max(mm.P_emp_random_stroma, 0.002):.3f}\nP$_{{RE}}$ = {fmt_p(mm.p_RE)};  I² = {mm.I2:.0%}")
    axT.text(0.04, (top + y) / 2, stat, fontsize=5.7, va="center", ha="left", color=INK, linespacing=1.4,
             transform=axT.get_yaxis_transform())
    y -= gap + 0.75
axC.axvline(0, color=GREY, lw=0.6, ls=(0, (2, 2)))
axC.set_yticks(yt); axC.set_yticklabels(ylab, fontsize=5.8)
axC.set_xlim(-0.8, 1.0); axC.set_ylim(y + gap + 0.2, 1.4)
axC.set_xlabel("partial Spearman ρ, SAP vs state signature | stromal + myeloid abundance")
axC.set_title("H2  pre-specified state associations replicate in both cohorts", loc="left", fontweight="bold")

axC.spines["left"].set_visible(False); axC.tick_params(axis="y", length=0)
for t, lab in zip(axC.get_yticklabels(), ylab):
    if lab == "FE meta":
        t.set_fontweight("bold")

# ================================================================== D  residual scatters
rowD = gridspec.GridSpecFromSubplotSpec(len(COH_ORDER), 3, subplot_spec=outer[2], wspace=0.28, hspace=0.42)
for i, c in enumerate(COH_ORDER):
    d = SC[SC.cohort == c]
    C = d[["Stroma_abundance", "Myeloid_abundance"]].values
    xres = rank_resid(d.SAP.values, C)
    for j, st in enumerate(["TAM_C1QC", "TAM_Inflammatory", "Mono_classical"]):
        ax = fig.add_subplot(rowD[i, j])
        if i == 0 and j == 0:
            panel_letter(ax, "D", dx=-0.2)
        yres = rank_resid(d[STATE[st]["sig"]].values, C)
        rho = np.corrcoef(xres, yres)[0, 1]
        rep = prim(st, c).r
        if abs(rho - rep) > 1e-6:
            raise RuntimeError(f"{c} {st}: re-derived rho {rho:.6f} != reported {rep:.6f}")
        xs, yss = xres / len(xres), yres / len(yres)
        kde = stats.gaussian_kde(np.vstack([xs, yss]))(np.vstack([xs, yss]))
        o = np.argsort(kde)
        base = COH[c]["color"]
        cm = LinearSegmentedColormap.from_list("c", ["#D9D9D9", base])
        ax.scatter(xs[o], yss[o], c=kde[o], cmap=cm, s=4, lw=0, rasterized=True)
        b = np.polyfit(xs, yss, 1)
        xx = np.linspace(xs.min(), xs.max(), 50)
        ax.plot(xx, np.polyval(b, xx), color=INK, lw=1.0)
        col = POS if STATE[st]["exp"] > 0 else NEG
        ax.text(0.03, 0.97, f"ρ = {rho:+.2f}", transform=ax.transAxes, fontsize=6.5, va="top",
                fontweight="bold", color=col)
        ax.text(0.97, 0.03, f"n = {len(xs)}", transform=ax.transAxes, fontsize=5.5, va="bottom", ha="right",
                color=GREY)
        ax.set_xticks([-0.4, 0, 0.4]); ax.set_yticks([-0.4, 0, 0.4])
        lim = 0.52
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        if i == 0:
            ax.set_title(STATE[st]["label"], fontweight="bold", color=col)
        if j == 0:
            ax.set_ylabel(f"{COH[c]['label']}\nstate (rank resid.)", fontsize=6.3)
        if i == len(COH_ORDER) - 1:
            ax.set_xlabel("SAP (rank residual)", fontsize=6.3)

# ================================================================== E / F
rowEF = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=outer[3], width_ratios=[0.07, 1.3, 1.0],
                                         height_ratios=[0.13, 1], wspace=0.55, hspace=0)

# ---- E: sensitivity heat map
axE = fig.add_subplot(rowEF[1, 1])
ana = [("H2 SAP vs state | stroma, myeloid", "Primary (stroma + myeloid)"),
       ("Sens: + epithelial abundance", "+ epithelial abundance"),
       ("Sens: raw SAP (no stromal adjustment)", "no stromal adjustment"),
       ("Sens: activated arm only", "activated arm only"),
       ("Sens: quiescent arm only (reversed)", "quiescent arm (reversed)")]
cols = [(st, c) for st in order for c in COH_ORDER]
M = np.full((len(ana), len(cols)), np.nan); Pm = M.copy()
for i, (a, _) in enumerate(ana):
    for j, (st, c) in enumerate(cols):
        r = P[(P.analysis == a) & (P.state == st) & (P.cohort == c)]
        if len(r):
            M[i, j], Pm[i, j] = r.r.iloc[0], r.p.iloc[0]
vmax = np.ceil(np.nanmax(np.abs(M)) * 10) / 10
im = axE.imshow(M, cmap=DIV, norm=TwoSlopeNorm(0, -vmax, vmax), aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if np.isfinite(M[i, j]):
            sig_ = Pm[i, j] < 0.05
            axE.text(j, i, f"{M[i, j]:+.2f}".replace("+", "") if M[i, j] >= 0 else f"{M[i, j]:.2f}",
                     ha="center", va="center", fontsize=5.4,
                     color=("white" if abs(M[i, j]) > 0.5 * vmax else INK) if sig_ else GREY,
                     fontstyle="normal" if sig_ else "italic")
axE.set_yticks(range(len(ana))); axE.set_yticklabels([b for _, b in ana], fontsize=6)
axE.set_xticks(range(len(cols)))
axE.set_xticklabels([COH[c]["label"][0] for _, c in cols], fontsize=5.8)
for k, st in enumerate(order):
    x0 = k * len(COH_ORDER) - 0.5
    if k:
        axE.axvline(x0, color="white", lw=2)
    axE.text(x0 + len(COH_ORDER) / 2, -0.62, STATE[st]["label"].replace("TAM ", "").replace("Classical ", "Cl. "),
             ha="center", va="bottom", fontsize=6, fontweight="bold",
             color=POS if STATE[st]["exp"] > 0 else NEG if STATE[st]["exp"] < 0 else GREY)
axE.axhline(0.5, color="white", lw=2)
for s_ in axE.spines.values():
    s_.set_visible(False)
axE.tick_params(length=0)
pe = axE.get_position()
fig.text(0.015, pe.y1 + 0.036, "E", fontsize=10, fontweight="bold", va="bottom")
fig.text(0.045, pe.y1 + 0.038, "Sensitivity analyses", fontsize=7.5, fontweight="bold", va="bottom")
cax = fig.add_axes([pe.x0 + pe.width * 0.55, pe.y0 - 0.052, pe.width * 0.45, 0.008])
cb = fig.colorbar(im, cax=cax, orientation="horizontal")
cb.set_label("partial Spearman ρ", fontsize=6, labelpad=1); cb.ax.tick_params(labelsize=5.5, length=2); cb.outline.set_lw(0.4)
axE.text(-0.5, -0.135, "A = ACRG, Y = Yonsei\ngrey italic: P ≥ 0.05", transform=axE.transAxes,
         fontsize=5.5, color=GREY, va="top")

# ---- F: discovery robustness
axF = fig.add_subplot(rowEF[1, 2])
dstates = ["Mono_classical", "TAM_Inflammatory", "TAM_C1QC", "TAM_Prolif"]
dlab = {"Mono_classical": "Classical monocyte", "TAM_Inflammatory": "TAM Inflammatory",
        "TAM_C1QC": "TAM C1QC", "TAM_Prolif": "TAM Prolif"}
models = [("Fig. 1E (reproduced)", "Fig. 1E model", INK, "o"),
          ("+ stromal fraction", "+ stromal fraction", "#7B4FA0", "s"),
          (">=100 myeloid cells", "≥100 myeloid cells", "#3A9A6E", "^")]
for k, st in enumerate(dstates):
    yb = len(dstates) - 1 - k
    loo = D[(D.analysis == "leave-one-tumour-out") & (D.state == st)]
    bt = D[(D.analysis == "bootstrap 95% CI") & (D.state == st)]
    if len(loo):
        axF.add_patch(FancyBboxPatch((loo.beta_min.iloc[0], yb - 0.38), loo.beta_max.iloc[0] - loo.beta_min.iloc[0],
                                     0.76, boxstyle="round,pad=0,rounding_size=0.05", fc="#E9E2D0", ec="none",
                                     zorder=0))
    if len(bt):
        axF.plot([bt.ci_lo.iloc[0], bt.ci_hi.iloc[0]], [yb + 0.27] * 2, color="#B08D3C", lw=0.9)
    for m, (a, lab, col, mk) in enumerate(models):
        r = D[(D.analysis == a) & (D.state == st)]
        if not len(r):
            continue
        r = r.iloc[0]
        yy = yb + 0.12 - 0.17 * m
        axF.plot([r.beta - 1.96 * r.se, r.beta + 1.96 * r.se], [yy, yy], color=col, lw=0.9)
        axF.plot(r.beta, yy, mk, ms=3.6, color=col, mfc=col if r.p < 0.05 else "white", mew=0.9, zorder=3)
axF.axvline(0, color=GREY, lw=0.6, ls=(0, (2, 2)))
axF.set_yticks(range(len(dstates))); axF.set_yticklabels([dlab[s] for s in dstates][::-1], fontsize=6.2)
axF.set_ylim(-0.6, len(dstates) - 0.4)
axF.set_xlabel("CLR composition β per SD of SAP")
pf = axF.get_position()
fig.text(pf.x0 - 0.145, pe.y1 + 0.036, "F", fontsize=10, fontweight="bold", va="bottom")
fig.text(pf.x0 - 0.115, pe.y1 + 0.038, f"Discovery robustness (scRNA-seq, n = {n_disc})", fontsize=7.5,
         fontweight="bold", va="bottom")
h = [Line2D([], [], marker=mk, color=col, ls="-", lw=0.9, ms=3.6, label=lab) for _, lab, col, mk in models]
h += [plt.Rectangle((0, 0), 1, 1, fc="#E9E2D0", ec="none", label="leave-one-out range"),
      Line2D([], [], color="#B08D3C", lw=0.9, label="bootstrap 95% CI")]
n100 = int(D[D.analysis == ">=100 myeloid cells"].n.iloc[0])
h[2].set_label(f"≥100 myeloid cells (n = {n100})")
h.append(Line2D([], [], marker="o", ls="", mfc="white", mec=GREY, ms=3.6, label="open symbol: P ≥ 0.05"))
axF.legend(handles=h, fontsize=5.4, loc="upper center", bbox_to_anchor=(0.4, -0.2), ncol=2,
           handlelength=1.6, columnspacing=0.8, handletextpad=0.4)

# ------------------------------------------------------------------ save
out = FIG / "Fig_SAP_external_validation"
for ext in ("pdf", "svg", "png"):
    fig.savefig(f"{out}.{ext}", dpi=600 if ext == "png" else None)
fig.savefig(f"{out}.tiff", dpi=600, pil_kwargs={"compression": "tiff_lzw"})
print(f"saved {out}.pdf/.svg/.png/.tiff")